# 02 — LLM Insight Generation

Validate the IONOS LLM integration for generating growth insights:
- Load analytics data from previous notebook
- Call IONOS Llama 3.3 with insight prompt
- Iterate on prompt quality
- Validate structured JSON output

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

IONOS_API_TOKEN = os.getenv('IONOS_API_TOKEN')
print(f'IONOS token loaded: {"yes" if IONOS_API_TOKEN else "NO — add to .env"}')

IONOS token loaded: yes


## 1. Load Analytics Data

Load the insights from `01_umami_ingest` notebook output.

In [2]:
from agent.storage import LocalStorage

store = LocalStorage(base_dir='../state')
insights_raw = store.read('insights.json')

if insights_raw is None:
    print('ERROR: Run 01_umami_ingest.ipynb first to generate insights.json')
else:
    print(f'Loaded insights: {insights_raw["website_analytics"]["visitors"]} visitors')
    print(f'Top pages: {len(insights_raw["website_analytics"]["top_pages"])}')
    print(f'Funnels: {list(insights_raw["website_analytics"]["event_funnels"].keys())}')

Loaded insights: 24 visitors
Top pages: 10
Funnels: ['wallet']


## 2. Connect to IONOS LLM

In [3]:
from agent.llm_client import LLMClient

llm = LLMClient(api_token=IONOS_API_TOKEN)

# Quick test
test = llm.chat([
    {'role': 'user', 'content': 'Say hello in exactly 5 words.'}
])
print(f'LLM response: {test["content"]}')
print(f'Tokens used: {test["usage"]}')

LLM response: Hello to my new friend.
Tokens used: {'prompt_tokens': 43, 'total_tokens': 50, 'completion_tokens': 7, 'prompt_tokens_details': None}


## 3. Insight Generation Prompt

Send analytics data to the LLM and ask for growth opportunities.

In [4]:
import json

# Load strategy defaults
from agent.models import Strategy, LLMAnalysis
strategy = Strategy()

insight_prompt = f"""You are a social media growth analyst for a technical blog (fretchen.eu).

The blog covers: {', '.join(strategy.content_pillars)}
Social channels: {', '.join(strategy.channels)}
Target audience: {strategy.target_audience}

Here is the website analytics data from the last 7 days:

Summary:
- Pageviews: {insights_raw['website_analytics']['pageviews']}
- Unique visitors: {insights_raw['website_analytics']['visitors']}
- Visits: {insights_raw['website_analytics']['visits']}
- Bounces: {insights_raw['website_analytics']['bounces']}

Top pages:
{json.dumps(insights_raw['website_analytics']['top_pages'][:10], indent=2)}

Top referrers:
{json.dumps(insights_raw['website_analytics']['top_referrers'][:10], indent=2)}

Tracked events (user engagement funnels):
{json.dumps(insights_raw['website_analytics']['top_events'][:10], indent=2)}

Based on this data, identify:
1. Which blog topics have the most visitor interest?
2. Where is traffic coming from? Any social media referrals?
3. Which pages would make the best social media content to share?
4. What content gaps exist — popular topics with no recent social posts?
5. Suggest 3-5 specific, actionable growth opportunities for Mastodon and Bluesky."""

print(f'Prompt length: {len(insight_prompt)} chars')
print('---')
print(insight_prompt[:500] + '...')

Prompt length: 1700 chars
---
You are a social media growth analyst for a technical blog (fretchen.eu).

The blog covers: Politische Ökonomie & Spieltheorie, Blockchain & Web3 (NFTs, x402, Smart Contracts), Quantencomputing & QML, AI-Tools & Infrastruktur
Social channels: mastodon, bluesky
Target audience: Tech-curious academics, developers, blockchain enthusiasts

Here is the website analytics data from the last 7 days:

Summary:
- Pageviews: 27
- Unique visitors: 24
- Visits: 25
- Bounces: 24

Top pages:
[
  {
    "path": ...


In [5]:
# Call LLM with structured output — response is a typed LLMAnalysis model
analysis = llm.structured_output(
    schema=LLMAnalysis,
    messages=[
        {'role': 'system', 'content': 'You are a data-driven social media growth analyst.'},
        {'role': 'user', 'content': insight_prompt}
    ],
)

print(f'Type: {type(analysis).__name__}')
print(f'Top topics: {analysis.top_topics}')
print(f'Pages for social: {len(analysis.best_pages_for_social)}')

Type: LLMAnalysis
Top topics: ['Blockchain & Web3', 'Quantencomputing & QML']
Pages for social: 3


## 4. Inspect Structured Output

In [6]:
# analysis is already a typed LLMAnalysis — no JSON parsing needed
print('=== Parsed Analysis ===')
print(f'\nTop topics: {analysis.top_topics}')
print(f'\nTraffic sources: {analysis.traffic_sources}')
print(f'\nBest pages for social:')
for page in analysis.best_pages_for_social:
    print(f'  - {page.url} — {page.reason}')
print(f'\nContent gaps: {analysis.content_gaps}')
print(f'\nGrowth opportunities:')
for opp in analysis.growth_opportunities:
    print(f'  - {opp}')

=== Parsed Analysis ===

Top topics: ['Blockchain & Web3', 'Quantencomputing & QML']

Traffic sources: ['github.com', 'fretchen.eu']

Best pages for social:
  - /amo/12/ — most visited page
  - /x402/ — high engagement
  - /blog/25/#allocating-risk-across-asset-classes — high engagement

Content gaps: ['AI-Tools & Infrastruktur', 'Politische Ökonomie & Spieltheorie']

Growth opportunities:
  - Create a Mastodon post about the latest developments in x402, including a brief summary and a link to the /x402/ page.
  - Share a Bluesky thread discussing the intersection of blockchain and quantum computing, including a link to the /quantum/amo/6/ page.
  - Host an AMA (Ask Me Anything) session on Mastodon, focusing on topics related to blockchain, quantum computing, and AI, and promote it on the blog and other social media channels.


## 5. Update Insights with Growth Opportunities

In [7]:
from datetime import datetime
from agent.models import Insights

# Update insights with LLM analysis
insights = Insights(**insights_raw)
insights.growth_opportunities = analysis.growth_opportunities
insights.last_analysis = datetime.now()

# Save enriched insights
store.write('insights.json', insights)

# Save typed LLM analysis (serialized via Pydantic)
store.write('llm_analysis.json', analysis)

print(f'Saved {len(insights.growth_opportunities)} growth opportunities')
print(f'Analysis timestamp: {insights.last_analysis}')

Saved 3 growth opportunities
Analysis timestamp: 2026-04-13 06:44:19.107967


In [8]:
llm.close()
print('Done — LLM insight generation validated.')

Done — LLM insight generation validated.
